# GxP-LLM quantization and evaluation

Attach `gxp-source`, `gxp-data`, and the output of a completed fine-tuning notebook. This notebook publishes GPTQ and AWQ models plus a complete, comparable evaluation for each variant.

In [ ]:
%pip install -q transformers accelerate sentence-transformers rouge-score nltk litellm wandb peft bitsandbytes
%pip install -q llmcompressor
%pip install -q gptqmodel #--no-build-isolation

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print(torch.cuda.get_device_name(0))

In [ ]:
import gc, json, os, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import wandb

root = Path("/kaggle/input/datasets/yuvalmehta")
print([p.name for p in root.iterdir()])

SOURCE_DIR = root / 'gxp-source'
DATA_DIR = root / 'gxp-data'

MODEL_PATH = Path('/kaggle/input/notebooks/yuvalmehta/gxpilot-train/gxp_train/merged_16bit')
m_root = Path('/kaggle/input/notebooks/yuvalmehta/gxpilot-train')
print([p.name for p in m_root.iterdir()])
assert (SOURCE_DIR / 'eval' / 'run.py').exists() and (DATA_DIR / 'train.jsonl').exists() and MODEL_PATH.exists(), 'Attach source, data, and fine-tuning notebook output.'
sys.path.insert(0, str(SOURCE_DIR))
OUTPUT_DIR = Path('/kaggle/working/gxp_quantization'); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JUDGE_MODEL = 'gemini/gemini-3.5-flash-lite'

def load_secret(name):
    value = UserSecretsClient().get_secret(name)
    if not value: raise RuntimeError(f'Add {name} as a Kaggle Secret.')
    os.environ[name] = value
load_secret('WANDB_API_KEY')
load_secret({'gemini/': 'GEMINI_API_KEY', 'groq/': 'GROQ_API_KEY', 'nvidia_nim/': 'NVIDIA_API_KEY'}[next(k for k in ('gemini/', 'groq/', 'nvidia_nim/') if JUDGE_MODEL.startswith(k))])
config = {'stage':'quantization','source_model':str(MODEL_PATH),'bits':4,'group_size':128,'judge_model':JUDGE_MODEL}
run = wandb.init(project='gxp-llm', name='quantize-4bit', job_type='quantize', tags=['gptq','awq'], config=config)
(OUTPUT_DIR / 'experiment_config.json').write_text(json.dumps(config, indent=2))

def calibration_texts(limit=128):
    examples = [json.loads(line) for line in (DATA_DIR / 'train.jsonl').read_text().splitlines()[:limit]]
    return [x['messages'][0]['content'] + '\n' + x['messages'][1]['content'] for x in examples]
CALIBRATION_TEXTS = calibration_texts()

In [ ]:
# GPTQ 4-bit
from gptqmodel import GPTQModel, QuantizeConfig
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True); tokenizer.pad_token = tokenizer.eos_token
gptq_path = OUTPUT_DIR / 'gptq-4bit'
quantize_config = QuantizeConfig(bits=4, group_size=128, desc_act=False, sym=True)
gptq_model = GPTQModel.load(str(MODEL_PATH), quantize_config, device_map='auto', trust_remote_code=True)
gptq_model.quantize(CALIBRATION_TEXTS); gptq_model.save_quantized(str(gptq_path)); tokenizer.save_pretrained(str(gptq_path))
del gptq_model; gc.collect(); torch.cuda.empty_cache()
print(f'GPTQ saved to {gptq_path}')

In [ ]:
import gc
import torch

from datasets import Dataset
from transformers import AutoModelForCausalLM

from llmcompressor import oneshot
from llmcompressor.modifiers.transform.awq import (
    AWQModifier,
    get_layer_mappings_from_model,
)
from llmcompressor.modifiers.quantization import QuantizationModifier


# ============================================================
# Output
# ============================================================

awq_path = OUTPUT_DIR / "awq-4bit"


# ============================================================
# Load model
# ============================================================

awq_model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True,
)


# ============================================================
# Calibration dataset
# ============================================================

calib_dataset = Dataset.from_dict({
    "text": CALIBRATION_TEXTS
})


# ============================================================
# Get Qwen3.5's official dynamic AWQ mappings
# ============================================================

all_mappings = get_layer_mappings_from_model(awq_model)

print(f"Total mappings: {len(all_mappings)}")

for i, mapping in enumerate(all_mappings):
    print(
        f"Mapping {i}: "
        f"{mapping.smooth_layer} -> {mapping.balance_layers}"
    )


# ============================================================
# IMPORTANT
#
# Qwen3.5 mappings:
#
#   Mapping 0 = full attention
#   Mapping 1 = GatedDeltaNet / linear attention
#   Mapping 2 = MLP gate/up
#   Mapping 3 = MLP up/down
#
# Exclude Mapping 1 because its calibration path invokes
# Qwen3_5GatedDeltaNet and causes:
#
#   forward() missing 1 required positional argument:
#   'hidden_states'
# ============================================================

awq_mappings = [
    all_mappings[0],   # full attention
    all_mappings[2],   # MLP gate/up
    all_mappings[3],   # MLP up/down
]

print(f"\nUsing {len(awq_mappings)} AWQ mappings")


# ============================================================
# AWQ + W4A16
# ============================================================

recipe = [
    AWQModifier(
        mappings=awq_mappings,
        duo_scaling="both",
    ),

    QuantizationModifier(
        scheme="W4A16_ASYM",
        targets=["Linear"],
        ignore=[
            "lm_head",

            # Do not quantize Qwen3.5 GatedDeltaNet projections.
            "re:.*linear_attn.*",
        ],
    ),
]


# ============================================================
# Run
# ============================================================

oneshot(
    model=awq_model,
    dataset=calib_dataset,
    recipe=recipe,
    output_dir=str(awq_path),
    max_seq_length=512,
    num_calibration_samples=len(CALIBRATION_TEXTS),
)


# ============================================================
# Save tokenizer
# ============================================================

tokenizer.save_pretrained(str(awq_path))


# ============================================================
# Cleanup
# ============================================================

del awq_model
gc.collect()
torch.cuda.empty_cache()

print(f"\nAWQ saved to: {awq_path}")

In [ ]:
# Identical full evaluation after every quantization method. GPTQ/AWQ already contain quantization config.
from eval.run import run_full_eval

for name, model_path in {'gptq_4bit': gptq_path, 'awq_4bit': awq_path}.items():
    results = run_full_eval(model_path=str(model_path), data_dir=str(DATA_DIR), judge_model=JUDGE_MODEL, output_dir=str(OUTPUT_DIR / f'eval_{name}'), load_in_4bit=False)
    for split, result in results.items():
        wandb.log({f'{name}/{split}/exact_match': result.metrics.get('exact_match', 0), f'{name}/{split}/rougeL_f1': result.metrics.get('rougeL', {}).get('rougeL_f1', 0), f'{name}/{split}/bleu': result.metrics.get('bleu', 0)})
        for category, scores in result.judge_scores.items():
            wandb.log({f'{name}/{split}/judge/{category}/{metric}': value for metric, value in scores.items()})
        if result.adversarial:
            wandb.log({f'{name}/{split}/{metric}': value for metric, value in result.adversarial.items() if isinstance(value, (int, float))})
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Keep GPTQ/AWQ weights in Kaggle output only; upload evaluation metadata/results, never model files, to W&B.
artifact = wandb.Artifact('quantization-evaluation', type='evaluation', metadata=config)
artifact.add_file(str(OUTPUT_DIR / 'experiment_config.json'))
for result_file in OUTPUT_DIR.glob('eval_*/full_results.json'):
    artifact.add_file(str(result_file), name=str(result_file.relative_to(OUTPUT_DIR)))
wandb.log_artifact(artifact)
wandb.finish(); print(f'Published Kaggle output: {OUTPUT_DIR}')